# Import Lib

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import SimpleRNN, Dense
from tensorflow.keras.losses import MeanSquaredError

In [33]:
df = pd.read_csv("D:/Intellibi/GenAI/DL/RNN bike Demand/data/bike.csv")

In [34]:
df.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [35]:
df.shape

(731, 16)

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     731 non-null    int64  
 1   dteday      731 non-null    object 
 2   season      731 non-null    int64  
 3   yr          731 non-null    int64  
 4   mnth        731 non-null    int64  
 5   holiday     731 non-null    int64  
 6   weekday     731 non-null    int64  
 7   workingday  731 non-null    int64  
 8   weathersit  731 non-null    int64  
 9   temp        731 non-null    float64
 10  atemp       731 non-null    float64
 11  hum         731 non-null    float64
 12  windspeed   731 non-null    float64
 13  casual      731 non-null    int64  
 14  registered  731 non-null    int64  
 15  cnt         731 non-null    int64  
dtypes: float64(4), int64(11), object(1)
memory usage: 91.5+ KB


### dteday should be datetime format. converting it into datetime 

In [37]:
df["dteday"] = pd.to_datetime(df["dteday"])
df = df.sort_values("dteday")

In [38]:
df["dteday"].dtype

dtype('<M8[ns]')

In [39]:
df.isnull().sum()

instant       0
dteday        0
season        0
yr            0
mnth          0
holiday       0
weekday       0
workingday    0
weathersit    0
temp          0
atemp         0
hum           0
windspeed     0
casual        0
registered    0
cnt           0
dtype: int64

In [40]:
df.duplicated().sum()

np.int64(0)

In [41]:
FEATURES = ["season","yr","mnth","holiday","weekday","workingday",
            "weathersit","temp","atemp","hum","windspeed"]

In [42]:
TARGET = "cnt"

# Train Test Split
* We don’t use train_test_split because it mixes (shuffles) the data.
In time series, order is important, so we split data in sequence — first part for training, later part for testing

In [43]:
split = int(len(df) * 0.8) # Total Records 731 = 731=80%=584 
train_df = df[:split]   # train df will be till 584 rows 
test_df = df[split:]  # remaining 146 records for test

In [44]:
print(f"Split: {split}\n traindf={train_df.shape}\n testdf={test_df.shape}")

Split: 584
 traindf=(584, 16)
 testdf=(147, 16)


# FEATURE SCALING

* neural networks (RNN, ANN, LSTM) work better when data is in a small, similar range (0 to 1).
* MinMaxScaler Converts all feature values → between 0 and 1(All values will be between same range (0–1))
* We fit the scaler only on training data so it learns scaling parameters like min and max. Then we apply the same transformation to test data to prevent data leakage and ensure a fair evaluation.

In [45]:
x_scaler = MinMaxScaler()
X_train = x_scaler.fit_transform(train_df[FEATURES])
X_test = x_scaler.transform(test_df[FEATURES])

y_scaler = MinMaxScaler()
y_train = y_scaler.fit_transform(train_df[[TARGET]])
y_test = y_scaler.transform(test_df[[TARGET]])


# SEQUENCE CREATION 
#### (WINDOW = 7 [Day1, Day2, Day3, Day4, Day5, Day6, Day7] → predict Day8)
Why Sequence Creation
* RNN learns from past data (time order)
* Instead of giving single row, we give multiple past rows together
* Here Day 1–7 → predict Day 8 (This group of past data = sequence)

We create sequences so the model can learn patterns from past data instead of just one data point.


In [46]:
window = 7

X_train_seq, y_train_seq = [], []
for i in range(len(X_train) - window):
    X_train_seq.append(X_train[i:i+window])
    y_train_seq.append(y_train[i+window])

X_test_seq, y_test_seq = [], []
for i in range(len(X_test) - window):
    X_test_seq.append(X_test[i:i+window])
    y_test_seq.append(y_test[i+window])

X_train_seq = np.array(X_train_seq)
y_train_seq = np.array(y_train_seq)

X_test_seq = np.array(X_test_seq)
y_test_seq = np.array(y_test_seq)

# RESHAPE CHECK (RNN FORMAT)

Here if we see the shape it got changed.

Because we group rows into windows, so we lose some rows at the beginning/end

* Formula : New size = Original size - window
* 584 - 7 = 577
* 147 - 7 = 140

In [47]:
print("Train shape:", X_train_seq.shape)  # (samples, timesteps, features)
print("Test shape:", X_test_seq.shape)

Train shape: (577, 7, 11)
Test shape: (140, 7, 11)


# MODEL BUILDING
64=neurons (More neurons = more learning capacity)

return_sequences=True(it sends full sequence output to next layer) if it is false Next RNN will fail

SimpleRNN(32) - It reads the sequence and keeps updating memory. This layer reads the sequence and extracts the final pattern from past data

Dense 1 - This layer gives the final prediction (bike count)


7 days data → RNN understands → Dense gives answer

64 neurons = more capacity to learn patterns
* Needs more power to:
* capture patterns
* understand trends
* learn relationships

32 neurons = less neurons, more focused learning
* Second layer gets already processed data
* refines patterns
* summarizes information




In [48]:
model = Sequential()

model.add(SimpleRNN(64, return_sequences=True,
                    input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])))
model.add(SimpleRNN(32))
model.add(Dense(1))

d:\Intellibi\GenAI\DL\RNN bike Demand\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


# COMPILE MODEL

* The compile step defines the optimizer and loss function. Adam is used to update weights efficiently, and Mean Squared Error is used to measure prediction error for regression tasks.

In [49]:
model.compile(
    optimizer="adam",
    loss=MeanSquaredError()
)

# MODEL TRAINING

model.fit() is used to train the model using training data. It runs for multiple epochs, updates weights using batches, and evaluates performance on validation data to monitor learning
* X_train_seq, y_train_seq - Model learns from this
* validation_data=(X_test_seq, y_test_seq) - Test data. Checks model performance on unseen data
* epochs=10 Number of times model learns from full dataset
* batch_size=32 - Data is split into small chunks. Instead of giving all data at once.(Number of samples processed at once)


In [50]:
history = model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_test_seq, y_test_seq),
    epochs=10,
    batch_size=32
)

Epoch 1/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - loss: 0.0684 - val_loss: 0.0610
Epoch 2/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0193 - val_loss: 0.0566
Epoch 3/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0171 - val_loss: 0.0635
Epoch 4/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0163 - val_loss: 0.0590
Epoch 5/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0154 - val_loss: 0.0592
Epoch 6/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0138 - val_loss: 0.0564
Epoch 7/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0147 - val_loss: 0.0582
Epoch 8/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0149 - val_loss: 0.0515
Epoch 9/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0122 - val_loss: 0.0444
Epoch 10/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0141 - val_loss: 0.0500


# MODEL EVALUATION
* model.predict() : Generates predictions from test data
* inverse_transform() : Converts scaled values back to original values
* RMSE : Measures error between actual and predicted

In [51]:
pred = model.predict(X_test_seq)

# inverse scaling
pred = y_scaler.inverse_transform(pred)  #pred = y_scaler.inverse_transform(pred)
y_test_seq = y_scaler.inverse_transform(y_test_seq) #Converts actual values also back to real scale

# RMSE
rmse = np.sqrt(np.mean((y_test_seq - pred) ** 2))
print("RMSE:", rmse)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
RMSE: 1773.1913604265656


# OVERFITTING / UNDERFITTING CHECK

* loss      → training error
* val_loss  → validation error

In [52]:
print("Final Training Loss:", history.history['loss'][-1])
print("Final Validation Loss:", history.history['val_loss'][-1])

Final Training Loss: 0.014110279269516468
Final Validation Loss: 0.04998679831624031


# SAVE MODEL

In [23]:
model.save("models/model.h5")
joblib.dump(x_scaler, "models/x_scaler.pkl")
joblib.dump(y_scaler, "models/y_scaler.pkl")

['models/y_scaler.pkl']

# PREDICTION (NEXT DAY)

In [24]:
# last 7 days data
last_data = df[FEATURES].tail(7)

In [ ]:
# Scales the input data using the same scaler used during training
X = x_scaler.transform(last_data)

In [ ]:
# Reshapes data into RNN format → (samples, timesteps, features)
#we are predicting only one case (next day) so sample=1
#Number of past time steps - 7
#Number of columns (inputs)
X = X.reshape(1, 7, len(FEATURES))

In [ ]:
# Predicts the next value using the trained model
pred = model.predict(X)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step


In [ ]:
# Converts prediction back to original scale (actual bike count)
pred = y_scaler.inverse_transform(pred)

In [29]:
print("Next Day Prediction:", pred[0][0])

Next Day Prediction: 2446.355
